# Retail Store Sales — Data Cleaning Project

## 1. Project Overview
This project focuses on assessing and improving the quality of a retail transactions dataset using Python and Pandas.

**Objectives**
- Explore the structure and quality of the raw dataset.
- Investigate missing values before selecting an imputation method.
- Recover missing product and price information using relationships observed in the data.
- Handle missing quantities and transaction totals with documented assumptions.
- Validate the cleaned dataset and export a reusable CSV file.

> **Important:** Imputed values are estimates, not recovered original source values. The assumptions and limitations are documented alongside the relevant steps.

## 2. Project Setup

Place the original file at `data/retail_store_sales.csv` relative to this project folder.  
The notebook uses relative paths so it can be run on another computer without editing a machine-specific file path.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "python":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

INPUT_FILE = DATA_DIR / "retail_store_sales.csv"
OUTPUT_FILE = OUTPUT_DIR / "Cleaned_Dataset.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 3. Load the Raw Dataset

In [2]:
INPUT_FILE = DATA_DIR / "retail_store_sales.csv"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_FILE}\n"
        "Make sure 'retail_store_sales.csv' exists inside the project's data folder."
    )

df = pd.read_csv(INPUT_FILE)

df_raw = df.copy(deep=True)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

Rows: 12,575
Columns: 11


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.50,10.00,185.00,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.00,9.00,261.00,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.50,2.00,43.00,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.50,9.00,247.50,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.50,7.00,87.50,Digital Wallet,Online,2022-10-02,False


## 4. Initial Data Exploration

In [3]:
display(df.sample(min(5, len(df)), random_state=42))
df.info()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
7315,TXN_1981490,CUST_17,Beverages,Item_25_BEV,41.00,4.00,164.00,Digital Wallet,In-store,2022-12-12,False
2119,TXN_1966034,CUST_13,Beverages,Item_20_BEV,33.50,4.00,134.00,Digital Wallet,In-store,2024-08-26,True
4629,TXN_3760555,CUST_16,Furniture,Item_11_FUR,20.00,2.00,40.00,Cash,In-store,2022-07-12,True
357,TXN_2151215,CUST_02,Butchers,Item_5_BUT,11.00,4.00,44.00,Cash,In-store,2023-11-24,False
5894,TXN_9019672,CUST_10,Patisserie,Item_19_PAT,32.00,3.00,96.00,Cash,Online,2024-09-15,False


<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.1+ MB


In [4]:
# Check how much is missing in each column to decide how to handle it
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
}).sort_values("Missing Values", ascending=False)

display(missing_summary)

,Missing Values,Missing %
Discount Applied,4199,33.39
Item,1213,9.65
Price Per Unit,609,4.84
Total Spent,604,4.80
Quantity,604,4.80
Transaction ID,0,0.00
Customer ID,0,0.00
Category,0,0.00
Payment Method,0,0.00
Location,0,0.00


In [5]:
# Descriptive statistics for numeric and categorical columns
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Transaction ID,12575,12575,TXN_6867343,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,12575,25,CUST_05,544,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,12575,8,Furniture,1591,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Item,11362,200,Item_2_BEV,126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price Per Unit,"11,966.00",NaN,NaN,NaN,23.37,10.74,5.00,14.00,23.00,33.50,41.00
Quantity,"11,971.00",NaN,NaN,NaN,5.54,2.86,1.00,3.00,6.00,8.00,10.00
Total Spent,"11,971.00",NaN,NaN,NaN,129.65,94.75,5.00,51.00,108.50,192.00,410.00
Payment Method,12575,3,Cash,4310,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,12575,2,Online,6354,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Transaction Date,12575,1114,2022-05-30,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Check duplicated transaction identifiers
duplicate_transaction_ids = df["Transaction ID"].duplicated().sum()
print(f"Duplicated Transaction IDs: {duplicate_transaction_ids:,}")

Duplicated Transaction IDs: 0


## 5. Data Cleaning Preparation

Work on a separate copy so the original DataFrame remains unchanged.  
The lookup tables below are built from known (non-missing) observations in the original dataset.

In [7]:
df_clean = df.copy(deep=True)

required_columns = {
    "Transaction ID", "Customer ID", "Category", "Item",
    "Price Per Unit", "Quantity", "Total Spent",
    "Payment Method", "Location", "Transaction Date",
    "Discount Applied"
}

missing_columns = required_columns.difference(df_clean.columns)
if missing_columns:
    raise ValueError(f"Expected columns are missing: {sorted(missing_columns)}")

original_row_count = len(df_clean)
original_columns = df_clean.columns.tolist()

## 6. Recover Missing Item Values

### 6.1 Check whether Category + Price Per Unit identifies one Item

Before using these columns as a lookup key, check whether any key combination maps to more than one distinct item. This is important because an ambiguous lookup could assign an incorrect product or multiply rows during a merge.

In [8]:
# Check that each Category + Price pair points to a single Item before using it as a lookup key
category_price_item_counts = (
    df_clean.dropna(subset=["Category", "Price Per Unit", "Item"])
    .groupby(["Category", "Price Per Unit"])["Item"]
    .nunique()
)

ambiguous_category_price_keys = category_price_item_counts[
    category_price_item_counts > 1
]

print(f"Ambiguous Category + Price keys: {len(ambiguous_category_price_keys):,}")
display(ambiguous_category_price_keys.head(10))

Ambiguous Category + Price keys: 0


Series([], Name: Item, dtype: int64)

In [10]:
# Fill missing Item using the Category + Price lookup (only where Item is NaN)
if not ambiguous_category_price_keys.empty:
    raise ValueError(
        "Category + Price Per Unit is not a unique product lookup key. "
        "Review ambiguous combinations before imputing Item."
    )

item_lookup = (
    df_clean.dropna(subset=["Category", "Price Per Unit", "Item"])
    [["Category", "Price Per Unit", "Item"]]
    .drop_duplicates()
)

# Map only rows where Item is missing.
item_key = pd.MultiIndex.from_frame(
    df_clean[["Category", "Price Per Unit"]]
)
item_map = item_lookup.set_index(
    ["Category", "Price Per Unit"]
)["Item"]

missing_item_before = df_clean["Item"].isna().sum()
item_candidates = pd.Series(item_key.map(item_map), index=df_clean.index)
df_clean["Item"] = df_clean["Item"].fillna(item_candidates)

print(f"Missing Item before: {missing_item_before:,}")
print(f"Missing Item after:  {df_clean['Item'].isna().sum():,}")

Missing Item before: 609
Missing Item after:  609


## 7. Recover Missing Price Per Unit

Missing prices can be recovered in two ways:

1. **Item + Category lookup:** if each pair has only one price, use it to fill the gap.
2. **Total Spent / Quantity:** if the lookup can't help, calculate the price from the transaction itself.

Note: the second method gives the effective price paid, which matches the list price only if Total Spent has no discount applied.

In [14]:
# Check that each Item + Category pair has a single price before using it as a lookup key
item_category_price_counts = (
    df_clean.dropna(subset=["Item", "Category", "Price Per Unit"])
    .groupby(["Item", "Category"])["Price Per Unit"]
    .nunique()
)

ambiguous_item_category_keys = item_category_price_counts[
    item_category_price_counts > 1
]

print(f"Ambiguous Item + Category keys: {len(ambiguous_item_category_keys):,}")
display(ambiguous_item_category_keys.head(10))

Ambiguous Item + Category keys: 0


Series([], Name: Price Per Unit, dtype: int64)

In [ ]:
# Build a price lookup from rows where Item, Category and Price are all known
price_lookup = (
    df_clean.dropna(subset=["Item", "Category", "Price Per Unit"])
    [["Item", "Category", "Price Per Unit"]]
    .drop_duplicates()
)

price_map = price_lookup.set_index(["Item", "Category"])["Price Per Unit"]
price_key = pd.MultiIndex.from_frame(df_clean[["Item", "Category"]])
price_candidates = pd.Series(price_key.map(price_map), index=df_clean.index)

missing_price_before = df_clean["Price Per Unit"].isna().sum()

# First choice: recover the recorded unit price from the Item + Category lookup
df_clean["Price Per Unit"] = df_clean["Price Per Unit"].fillna(price_candidates)
missing_price_after_lookup = df_clean["Price Per Unit"].isna().sum()

# Fallback: infer the effective unit price from Total Spent / Quantity
can_infer_effective_price = (
    df_clean["Price Per Unit"].isna()
    & df_clean["Total Spent"].notna()
    & df_clean["Quantity"].notna()
    & df_clean["Quantity"].ne(0)
)

df_clean.loc[can_infer_effective_price, "Price Per Unit"] = (
    df_clean.loc[can_infer_effective_price, "Total Spent"]
    / df_clean.loc[can_infer_effective_price, "Quantity"]
)

print(f"Missing Price Per Unit before: {missing_price_before:,}")
print(f"Recovered from Item + Category lookup: {missing_price_before - missing_price_after_lookup:,}")
print(f"Inferred from Total Spent / Quantity: {can_infer_effective_price.sum():,}")
print(f"Missing Price Per Unit after: {df_clean['Price Per Unit'].isna().sum():,}")

Missing Price Per Unit before: 609
Recovered from Item + Category lookup: 0
Inferred from Total Spent / Quantity: 609
Missing Price Per Unit after: 0


## 8. Second Pass: Recover Item Values Using the Recovered Prices

Some rows may initially have both `Item` and `Price Per Unit` missing. After recovering the price from the `Item + Category` relationship where possible, perform a second item lookup using `Category + Price Per Unit`.

In [17]:
item_key_after_price = pd.MultiIndex.from_frame(
    df_clean[["Category", "Price Per Unit"]]
)
item_candidates_second_pass = pd.Series(
    item_key_after_price.map(item_map),
    index=df_clean.index
)

missing_item_before_second_pass = df_clean["Item"].isna().sum()
df_clean["Item"] = df_clean["Item"].fillna(item_candidates_second_pass)

print(f"Additional Item values recovered: "
      f"{missing_item_before_second_pass - df_clean['Item'].isna().sum():,}")
print(f"Remaining missing Item values: {df_clean['Item'].isna().sum():,}")

Additional Item values recovered: 609
Remaining missing Item values: 0


## 9. Impute Missing Quantity Values

`Quantity` represents a count of units, so the mode is used as a product-level estimate rather than the median. This keeps imputed values aligned with whole-unit quantities.

The mode is calculated from observed quantities for each item. If an item has no observed quantity, the value remains missing and is reported for review rather than filled with an arbitrary global value.

In [18]:
quantity_mode = (
    df_clean.dropna(subset=["Item", "Quantity"])
    .groupby("Item")["Quantity"]
    .agg(lambda values: values.mode().iloc[0] if not values.mode().empty else np.nan)
)

missing_quantity_before = df_clean["Quantity"].isna().sum()
df_clean["Quantity"] = df_clean["Quantity"].fillna(
    df_clean["Item"].map(quantity_mode)
)

print(f"Missing Quantity before: {missing_quantity_before:,}")
print(f"Missing Quantity after:  {df_clean['Quantity'].isna().sum():,}")

Missing Quantity before: 604
Missing Quantity after:  0


## 10. Handle Missing Total Spent

For rows where `Total Spent` is missing but both price and quantity are available, calculate the line total as `Price Per Unit × Quantity`.

**Assumption and limitation:** This calculation assumes that `Total Spent` is the amount represented by the unit price multiplied by quantity. If the source system stores the amount after discounts while `Price Per Unit` is the pre-discount price, this formula may not reproduce the actual transaction total. The dataset does not provide enough information here to reconstruct an unknown discount amount, so this assumption should be verified with the data owner before business use.

In [19]:
missing_total_before = df_clean["Total Spent"].isna().sum()

can_calculate_total = (
    df_clean["Total Spent"].isna()
    & df_clean["Price Per Unit"].notna()
    & df_clean["Quantity"].notna()
)

df_clean.loc[can_calculate_total, "Total Spent"] = (
    df_clean.loc[can_calculate_total, "Price Per Unit"]
    * df_clean.loc[can_calculate_total, "Quantity"]
)

print(f"Total Spent values calculated: {can_calculate_total.sum():,}")
print(f"Missing Total Spent after calculation: {df_clean['Total Spent'].isna().sum():,}")

Total Spent values calculated: 604
Missing Total Spent after calculation: 0


## 11. Handle Missing Discount Applied Values

Missing discount information is labeled `Unknown` instead of being converted to `False`. A missing value does not prove that a discount was not applied.

In [20]:
missing_discount_before = df_clean["Discount Applied"].isna().sum()
df_clean["Discount Applied"] = df_clean["Discount Applied"].fillna("Unknown")

print(f"Discount values labeled Unknown: {missing_discount_before:,}")
display(df_clean["Discount Applied"].value_counts(dropna=False))

Discount values labeled Unknown: 4,199


Discount Applied
True       4219
Unknown    4199
False      4157
Name: count, dtype: int64

## 12. Correct Data Types

Convert the transaction date to datetime and quantity to Pandas' nullable integer type. The nullable `Int64` type preserves missing values if any quantity could not be imputed.

In [23]:
# Dates in the file look like 2024-04-08, so the format is %Y-%m-%d
df_clean["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],     # read from the raw column, since df_clean's is now all NaT
    format="%Y-%m-%d",
    errors="coerce"
)

df_clean["Quantity"] = pd.to_numeric(
    df_clean["Quantity"], errors="coerce"
).round().astype("Int64")

df_clean["Discount Applied"] = df_clean["Discount Applied"].astype("string")

print(f"Dates that failed to parse: {df_clean['Transaction Date'].isna().sum():,}")
df_clean.dtypes

Dates that failed to parse: 0


Transaction ID                 str
Customer ID                    str
Category                       str
Item                           str
Price Per Unit             float64
Quantity                     Int64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
Discount Applied            string
dtype: object

## 13. Final Data Quality Assessment

Do not assume that a dataset is fully clean just because missing values have been reduced. Review missingness, duplicate identifiers, data types, and basic business-rule checks.

In [24]:
final_missing_summary = pd.DataFrame({
    "Missing Values": df_clean.isna().sum(),
    "Missing %": (df_clean.isna().mean() * 100).round(2)
})

display(final_missing_summary)

,Missing Values,Missing %
Transaction ID,0,0.00
Customer ID,0,0.00
Category,0,0.00
Item,0,0.00
Price Per Unit,0,0.00
Quantity,0,0.00
Total Spent,0,0.00
Payment Method,0,0.00
Location,0,0.00
Transaction Date,0,0.00


In [25]:
# Check duplicates and basic validity rules
quality_checks = {
    "Row count unchanged": len(df_clean) == original_row_count,
    "Column names unchanged": df_clean.columns.tolist() == original_columns,
    "Transaction ID is unique": df_clean["Transaction ID"].is_unique,
    "No duplicate transaction IDs": df_clean["Transaction ID"].duplicated().sum() == 0,
    "Quantity is positive when present": (
        df_clean["Quantity"].dropna().gt(0).all()
    ),
    "Price Per Unit is non-negative when present": (
        df_clean["Price Per Unit"].dropna().ge(0).all()
    ),
    "Total Spent is non-negative when present": (
        df_clean["Total Spent"].dropna().ge(0).all()
    ),
    "Transaction Date parsed successfully": (
        df_clean["Transaction Date"].notna().all()
    ),
}

quality_report = pd.DataFrame(
    [{"Check": name, "Passed": bool(result)} for name, result in quality_checks.items()]
)

display(quality_report)

,Check,Passed
0,Row count unchanged,True
1,Column names unchanged,True
2,Transaction ID is unique,True
3,No duplicate transaction IDs,True
4,Quantity is positive when present,True
5,Price Per Unit is non-negative when present,True
6,Total Spent is non-negative when present,True
7,Transaction Date parsed successfully,True


In [26]:
# Check whether any missing values remain and identify them explicitly.
remaining_missing = df_clean.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if remaining_missing.empty:
    print("No missing values remain.")
else:
    print("Some values remain missing and require review:")
    display(remaining_missing)

No missing values remain.


In [27]:
# A failed check should be investigated; do not silently ignore it.
failed_checks = quality_report.loc[~quality_report["Passed"], "Check"].tolist()

if failed_checks:
    print("Review required for:", failed_checks)
else:
    print("All defined quality checks passed.")

All defined quality checks passed.


## 14. Compare Before and After

This summary shows the impact of the cleaning process without hiding any remaining missing values.

In [28]:
before_missing = df_raw.isna().sum().sum()
after_missing = df_clean.isna().sum().sum()

cleaning_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Missing cells",
        "Duplicate Transaction IDs"
    ],
    "Before": [
        len(df_raw),
        df_raw.shape[1],
        before_missing,
        df_raw["Transaction ID"].duplicated().sum()
    ],
    "After": [
        len(df_clean),
        df_clean.shape[1],
        after_missing,
        df_clean["Transaction ID"].duplicated().sum()
    ]
})

display(cleaning_summary)

,Metric,Before,After
0,Rows,12575,12575
1,Columns,11,11
2,Missing cells,7229,0
3,Duplicate Transaction IDs,0,0


## 15. Export the Cleaned Dataset

Export the cleaned dataset only after reviewing the validation report and any remaining missing values.

In [29]:
df_clean.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset exported to: {OUTPUT_FILE}")
print(f"Final shape: {df_clean.shape}")

Cleaned dataset exported to: e:\Study\Data Anlaysing\Project\Retail\Retail_Store_Sales_Analysis\output\Cleaned_Dataset.csv
Final shape: (12575, 11)


## 16. Key Decisions and Limitations

- **Item recovery:** Uses the observed `Category + Price Per Unit` relationship only after checking that the key is unambiguous.
- **Price recovery:** Uses the observed `Item + Category` relationship first. If that cannot fill the price, `Total Spent / Quantity` is used as a fallback when both values exist. This fallback estimates an effective unit amount and may reflect a discount.
- **Quantity imputation:** Uses the most frequent observed quantity for the same item. This is an estimate and may not represent the missing transaction's true quantity.
- **Total Spent:** Calculated from price and quantity only when both are available. This assumes the total is consistent with the unit price multiplied by quantity; verify whether discounts affect this field.
- **Discount Applied:** Missing values are retained as `Unknown`, not interpreted as `False`.
- **Remaining missing values:** Are reported for further investigation rather than filled with unsupported assumptions.